In [1]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine,\
    SingleStateAnsatz,create_single_machine,\
        create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,\
        NESFermionHopRule,compute_qgt,sampler_info
import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
import time

# ========== 你原有全局参数（直接复用） ==========
# 单系统希尔伯特空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
K = 3  # NES 扩展副本数
hi_ext = hi ** K  # 扩展希尔伯特空间
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
single_edges = ((0, 1), (2, 3))  # 费米子跃迁边
g = nk.graph.Graph(edges=single_edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)

total_ansatz = NESTotalAnsatz(4,K,12,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef,total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: Prefer the new nk.driver.VMC_SR over VMC which supports minSR and SPRING.

In [2]:
from pyscf import gto, scf, fci
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能：{exc:.4f} eV")
# ===================== NetKet 哈密顿量和采样器 =====================
ha = nkx.operator.from_pyscf_molecule(mol)

hi = nkx.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV


/var/folders/8x/k_m4pmb11437ktb_r6tjzt2c0000gn/T/ipykernel_47537/751071632.py:20: DeprecationWarning: netket.experimental.hilbert.SpinOrbitalFermions is deprecated: use netket.hilbert.SpinOrbitalFermions (netket >= 3.12)
  hi = nkx.hilbert.SpinOrbitalFermions(


In [3]:
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER =400
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
Natural_Grad = True


total_ansatz = NESTotalAnsatz(4,K,12,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef,total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    
    

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)

ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)  # 转为jax数组（关键修复）

nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=16,
    sweep_size=20
)


# 采样器状态初始化（替代原 init_sampler_state）
sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)

# ==================== 训练循环（仅替换采样部分） ====================
print("\n" + "="*60)
print("开始多链 NES-VMC 训练 (NetKet 自定义采样器 + 朴素梯度下降)")
print("="*60)
print(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha")

history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_2st': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[],
    'log_Psi_mean':[],
    'log_Psi_min':[],
    'log_Psi_max':[],
    'grad_norm':[],
}

start_time = time.time()
for step in range(N_ITER):
    # 2. 正式采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine, parameters=total_params, 
        state=sampler_state, chain_length=N_SAMPLES_PER_CHAIN
    )
        # 3. 维度重塑，适配梯度函数输入
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, 4)
    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 total_matrix_machine=total_matrix_machine,
                                                 total_machine=total_machine,
                                                 single_machine_list=single_machine_list,
                                                 total_params=total_params,
                                                 x_batch=samples.reshape(-1,K,4))
    #grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    grad_flat , grad_unravel_fn = ravel_pytree(grad)
    if Natural_Grad == True:
        #grad_flat , grad_unravel_fn = ravel_pytree(grad)
        qgt_reg, unravel_fn = compute_qgt(total_machine, total_params, samples.reshape(-1,K,4), diag_shift=0.1)
        
        # # 自然梯度求解
        natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
        natural_grad = grad_unravel_fn(natural_grad_flat)
        grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    
    
    log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
    #eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
    eig_vals, eig_vecs = jnp.linalg.eig(E_L_mean)
    sort_idx = jnp.argsort(eig_vals.real)
    eig_vals, eig_vecs = eig_vals[sort_idx], eig_vecs[:, sort_idx]
    
    
    grad_norm = jnp.linalg.norm(grad_flat)
    
    
    history['step'].append(step)
    history['E_Lmatrix'].append(E_L_mean)
    history['samples'].append(samples)
    history['loss'].append(loss_mean)
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    history['log_Psi_min'].append(log_Psi_batch.min())
    history['log_Psi_max'].append(log_Psi_batch.max())
    history['grad_norm'].append(grad_norm)
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['energy_2st'].append(eig_vals[2])
    history['params'].append(total_params)
    # 5. 记录历史
    if step % 50 == 0 or step == N_ITER - 1:
        # --------------------- 【NES-VMC 监控模板】直接用 ---------------------
        # 1. 监控 log_Psi
        #log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
        print(f"log_Psi: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")

        # 2. 监控梯度范数
        
        print(f"grad norm = {grad_norm:.4f}")
        print(f"Step {step:3d} | Loss: {loss_mean}|0st能量={eig_vals[0]:.8f} Ha｜1st能量={eig_vals[1]:.8f} Ha｜2st能量={eig_vals[2]:.8f} Ha")
        # print(f'grad={grad_flat[30:31]}')
        print('#-----------------------------------------#')


end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
# 最终结果
print("\n" + "="*60)
print(f"训练完成!")
print("="*60)


开始多链 NES-VMC 训练 (NetKet 自定义采样器 + 朴素梯度下降)
基态能量=-1.01546825 Ha| 第一激发态能量=-0.87542794 Ha| 第二激发态能量=-0.42938376 Ha
log_Psi: mean=0.862-0.641j | min=0.371-2.245j | max=1.417+0.951j
grad norm = 2.9905
Step   0 | Loss: -1.9073312710365369|0st能量=-0.90931511-0.00131068j Ha｜1st能量=-0.71145162+0.00288677j Ha｜2st能量=-0.28656455-0.00041384j Ha
#-----------------------------------------#
log_Psi: mean=1.322-0.620j | min=0.282-2.110j | max=1.610+0.958j
grad norm = 0.0702
Step  50 | Loss: -2.272831560542264|0st能量=-1.01359245+0.00003805j Ha｜1st能量=-0.87471065-0.00053534j Ha｜2st能量=-0.38452846+0.00010333j Ha
#-----------------------------------------#
log_Psi: mean=1.856-1.011j | min=0.857-2.574j | max=2.111+0.563j
grad norm = 0.1005
Step 100 | Loss: -2.295495830909905|0st能量=-1.01477883+0.00024887j Ha｜1st能量=-0.87518010-0.00000706j Ha｜2st能量=-0.40553690+0.00001184j Ha
#-----------------------------------------#
log_Psi: mean=2.967-1.440j | min=1.410-3.061j | max=3.158+0.134j
grad norm = 0.0754
Step 150 | Loss:

In [ ]:
Natural_Grad = True

$$
\begin{align*}
\Psi(\mathbf{x})^{-1}\hat{\mathcal{H}}\Psi(\mathbf{x})
&= \mathrm{Tr}\left[ \Psi^{-1}(\mathbf{x})\hat{H}\Psi(\mathbf{x}) \right]
\end{align*}
$$

In [4]:
import pickle
import os  # 加上这个
# 自动创建 data 文件夹（关键修复）
os.makedirs('./data', exist_ok=True)
if Natural_Grad == True:
    print('保存自然梯度历史记录')
    # 保存 history
    with open('./data/history_natural_gradient_K3.pkl', 'wb') as f:
        pickle.dump(history, f)
else:
    # 保存 history
    with open('./data/history_plain_gradient_K3.pkl', 'wb') as f:
        pickle.dump(history, f)

print("保存成功！")

保存自然梯度历史记录
保存成功！


In [ ]:
import pickle
history_natural= pickle.load(open('./data/history_natural_gradient_K3.pkl', 'rb'))
history_plain= pickle.load(open('./data/history_plain_gradient_K3.pkl', 'rb'))


In [ ]:
history_natural['params'][0]

In [ ]:
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from NES_VMC import E_fcis

# 创建 2行1列 的子图
fig, axs = plt.subplots(3, 3, figsize=(12, 9))
fig.suptitle('Natural Excited State-VMC for $H_2$ K=3 ')
# 第一个子图
axs[0,0].plot(history_natural['energy_0st'],color='orange',label='natural gradient')
axs[0,0].plot(history_plain['energy_0st'],color='blue',label='plain gradient')
axs[0,0].hlines(E_fcis[0],0,len(history_natural['energy_0st']),linestyle='--',color='red')
axs[0,0].set_title('0st Energy')
axs[0,0].set_ylabel('energy')
axs[0,0].set_xlabel('step')
axs[0,0].set_xlabel('step')
axs[0,0].legend()

axs[0,1].plot(history_natural['energy_1st'],color='orange',label='natural gradient')
axs[0,1].plot(history_plain['energy_1st'],color='blue',label='plain gradient')
axs[0,1].hlines(E_fcis[1],0,len(history_natural['energy_1st']),linestyle='--',color='red')
axs[0,1].set_title('1st Energy')
axs[0,1].set_ylabel('energy')
axs[0,1].set_xlabel('step')      
axs[0,1].legend()

axs[0,2].plot(history_natural['energy_2st'],color='orange',label='natural gradient')
axs[0,2].plot(history_plain['energy_2st'],color='blue',label='plain gradient')
axs[0,2].hlines(E_fcis[2],0,len(history_natural['energy_2st']),linestyle='--',color='red')
axs[0,2].set_title('2st Energy')
axs[0,2].set_ylabel('energy')
axs[0,2].set_xlabel('step')      
axs[0,2].legend()


# # 第二个子图
axs[1,0].plot(history_natural['energy_0st']-E_fcis[0],color='orange',label='natural gradient')    
axs[1,0].set_title('0st Energy Error')
axs[1,0].set_xlabel('step')
axs[1,0].set_ylabel('energy')
axs[1,0].legend()



axs[1,1].plot(history_natural['energy_1st']-E_fcis[1],color='orange',label='natural gradient')    
axs[1,1].set_title('1st Energy Error')
axs[1,1].set_xlabel('step')
axs[1,1].set_ylabel('energy')
axs[1,1].legend()

axs[1,2].plot(history_natural['energy_2st']-E_fcis[2],color='orange',label='natural gradient')    
axs[1,2].set_title('2st Energy Error')
axs[1,2].set_xlabel('step')
axs[1,2].set_ylabel('energy')
axs[1,2].legend()


# # 第二个子图
axs[2,0].plot(history_natural['loss'],color='orange',label='natural gradient')  
axs[2,0].plot(history_plain['loss'],color='blue',label='plain gradient')
axs[2,0].set_title('loss')
axs[2,0].set_xlabel('step')
axs[2,0].set_ylabel('loss')
axs[2,0].legend()

axs[2,1].plot(history_natural['grad_norm'],color='orange',label='natural gradient')  
axs[2,1].plot(history_plain['grad_norm'],color='blue',label='plain gradient')
axs[2,1].set_title('grad_norm')
axs[2,1].set_xlabel('step')
axs[2,1].set_ylabel('grad_norm')
axs[2,1].legend()

axs[2,2].plot(history_natural['energy_0st'],color='orange')
axs[2,2].plot(history_natural['energy_1st'],color='orange')
axs[2,2].plot(history_natural['energy_2st'],color='orange')
axs[2,2].hlines(E_fcis[0],0,len(history_natural['energy_0st']),linestyle='--',color='red',label='0st Energy FCI')
axs[2,2].hlines(E_fcis[1],0,len(history_natural['energy_1st']),linestyle='--',color='red',label='1st Energy FCI')
axs[2,2].hlines(E_fcis[2],0,len(history_natural['energy_2st']),linestyle='--',color='red',label='2st Energy FCI')
axs[2,2].set_title('Energy')
axs[2,2].set_xlabel('step')
axs[2,2].set_ylabel('energy')
axs[2,2].legend()

plt.tight_layout()  # 自动调整间距
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from NES_VMC import E_fcis

# 创建 2行1列 的子图
fig, axs = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('NES-VMC for $H_2$ K=3 ')
# 第一个子图
axs[0].plot(history_natural['energy_0st'],color='orange',label='natural gradient')
axs[0].plot(history_plain['energy_0st'],color='blue',label='plain gradient')
axs[0].hlines(E_fcis[0],0,len(history_natural['energy_0st']),linestyle='--',color='red')
axs[0].set_title('0st Energy')
axs[0].set_ylabel('energy')
axs[0].set_xlabel('step')
axs[0].legend()

axs[1].plot(history_natural['energy_1st'],color='orange',label='natural gradient')
axs[1].plot(history_plain['energy_1st'],color='blue',label='plain gradient')
axs[1].hlines(E_fcis[1],0,len(history_natural['energy_1st']),linestyle='--',color='red')
axs[1].set_title('1st Energy')
axs[1].set_ylabel('energy')
axs[1].set_xlabel('step')      
axs[1].legend()

axs[2].plot(history_natural['energy_2st'],color='orange',label='natural gradient')
axs[2].plot(history_plain['energy_2st'],color='blue',label='plain gradient')
axs[2].hlines(E_fcis[2],0,len(history_natural['energy_2st']),linestyle='--',color='red')
axs[2].set_title('2st Energy')
axs[2].set_ylabel('energy')
axs[2].set_xlabel('step')      
axs[2].legend()


# 第二个子图
axs[3].plot(history_natural['loss'],color='orange',label='natural gradient')            
axs[3].plot(history_plain['loss'],color='blue',label='plain gradient')
axs[3].set_title('loss')
axs[3].set_xlabel('step')
axs[3].set_ylabel('energy')
axs[3].legend()

# 第三个子图
axs[4].plot(history_natural['grad_norm'],color='orange',label='natural gradient')
axs[4].plot(history_plain['grad_norm'],color='blue',label='plain gradient')
axs[4].set_title('grad_norm')
axs[4].set_xlabel('step')
axs[4].set_ylabel('grad_norm')
axs[4].legend()

plt.tight_layout()  # 自动调整间距
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 创建 2行1列 的子图
fig, axs = plt.subplots(1, 5, figsize=(12, 3))
fig.suptitle('NES-VMC for $H_2$ K=3 Natural Gradient Descent')
# 第一个子图
axs[0].plot(history['energy_0st'])
axs[0].hlines(E_fcis[0],0,len(history['energy_0st']),linestyle='--',color='red')
axs[0].set_title('0st Energy')
axs[0].set_ylabel('energy')
axs[0].set_xlabel('step')

axs[1].plot(history['energy_1st'])
axs[1].hlines(E_fcis[1],0,len(history['energy_1st']),linestyle='--',color='red')
axs[1].set_title('1st Energy')
axs[1].set_ylabel('energy')
axs[1].set_xlabel('step')

axs[2].plot(history['energy_2st'])
axs[2].hlines(E_fcis[2],0,len(history['energy_2st']),linestyle='--',color='red')
axs[2].set_title('2st Energy')
axs[2].set_ylabel('energy')
axs[2].set_xlabel('step')

# 第二个子图
axs[3].plot(history['loss'])
axs[3].set_title('loss')
axs[3].set_xlabel('step')
axs[3].set_ylabel('energy')

# 第三个子图
axs[4].plot(history['grad_norm'])
axs[4].set_title('grad_norm')
axs[4].set_xlabel('step')
axs[4].set_ylabel('grad_norm')

plt.tight_layout()  # 自动调整间距
plt.show()

剖析为啥有毛病 

In [ ]:
history['energy_0st'][40:45]

In [ ]:
from collections import Counter
import numpy as np
def sampler_info(samples:jnp.array,K:int):
    test_samples = np.array(samples.reshape(-1, 4*K))
    count = Counter(tuple(each_row.tolist()) for each_row in test_samples)
    for tpl, count_ in count.items():
        print(f"元组 {tpl} 出现了 {count_} 次")
    return count


sampler_info(history['samples'][100],K)

In [ ]:
sampler_info(history['samples'][102],K)

In [ ]:

print(f'我当时保存的: loss: {history["loss"][102]:.3f}|energy_0st: {history["energy_0st"][102]:.3f}|grad_norm: {history["grad_norm"][102]:.3f}')
test_samples = history['samples'][102]
test_params = history['params'][101]
# 3. 计算能量和自然梯度（逻辑和原代码一致）
grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                total_matrix_machine=total_matrix_machine,
                                                total_machine=total_machine,
                                                single_machine_list=single_machine_list,
                                                total_params=test_params,
                                                x_batch=test_samples.reshape(-1,K,4))

grad_flat , grad_unravel_fn = ravel_pytree(grad)
grad_norm = jnp.linalg.norm(grad_flat)

eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
print(f'我基于当时的 Samples 尝试复现:')
print(f"grad_norm: {grad_norm:.3f}|loss_mean: {loss_mean:.3f}|energy_0st: {eig_vals[0]:.3f}")
print(E_L_mean)

In [ ]:
from NES_VMC import NES_loss_energy

trace, E_L = NES_loss_energy(ha=ha,
                total_matrix_machine=total_matrix_machine,
                single_machine_list=single_machine_list,
                total_params=total_params,
                x=test_samples.reshape(-1,2,4))

trace

In [ ]:
test_samples.reshape(-1,2,4)[0]

In [ ]:
E_L[0]

In [ ]:
def NES_loss_energy(ha, total_matrix_machine,single_machine_list,total_params, x):
    log_M = total_matrix_machine(total_params,x)
    Psi_Matrix = jnp.exp(log_M)
    # 添加正则化项，防止矩阵奇异
    #Psi_Matrix += 1e-6 * jnp.eye(Psi_Matrix.shape[0])
    H_psi_x = Ham_Psi(ha,single_machine_list,total_params,x)
    Psi_Matrix_inv = jnp.linalg.solve(Psi_Matrix, H_psi_x)
    return jnp.real(jnp.trace(Psi_Matrix_inv, axis1=-2, axis2=-1)), Psi_Matrix_inv

In [ ]:
total_matrix_machine(total_params, history['samples'][41].reshape(-1,K,4))